# Euclid Q1 NISP

In [7]:
import re
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.nddata import Cutout2D
from astropy.utils.data import download_file
from astropy.wcs import WCS
from astropy import units as u

import pyvo as vo
import sep

# Copy-on-write is more performant and avoids unexpected modifications of the original DataFrame.
pd.options.mode.copy_on_write = True

In [3]:
search_radius = 5 * u.arcmin
coord = SkyCoord.from_name('HUDF')
print(coord)

<SkyCoord (ICRS): (ra, dec) in deg
    (53.1625, -27.79139)>


In [4]:
irsa_service= vo.dal.sia2.SIA2Service('https://irsa.ipac.caltech.edu/SIA')
image_table = irsa_service.search(pos=(coord, search_radius), collection='euclid_DpdMerBksMosaic',instrument='NISP',data_type='image')
df_im_irsa=image_table.to_table().to_pandas().reset_index()
df_im_euclid=df_im_irsa[ (df_im_irsa['dataproduct_subtype']=='science') & (df_im_irsa['energy_bandpassname']=='Y')].reset_index()
df_im_euclid

,level_0,index,s_ra,s_dec,facility_name,instrument_name,dataproduct_subtype,calib_level,dataproduct_type,energy_bandpassname,...,t_resolution,t_xel,obs_publisher_did,s_fov,em_xel,pol_states,pol_xel,cloud_access,o_ucd,upload_row_id
0,7,7,53.550627,-27.999986,Euclid,NISP,science,3,image,Y,...,NaN,<NA>,ivo://irsa.ipac/euclid_DpdMerBksMosaic?1020441...,0.533333,<NA>,,<NA>,"{""aws"": {""bucket_name"": ""nasa-irsa-euclid-q1"",...",,1
1,21,21,53.310481,-27.499986,Euclid,NISP,science,3,image,Y,...,NaN,<NA>,ivo://irsa.ipac/euclid_DpdMerBksMosaic?1020448...,0.533333,<NA>,,<NA>,"{""aws"": {""bucket_name"": ""nasa-irsa-euclid-q1"",...",,1
2,39,39,52.986936,-27.999986,Euclid,NISP,science,3,image,Y,...,NaN,<NA>,ivo://irsa.ipac/euclid_DpdMerBksMosaic?1020441...,0.533333,<NA>,,<NA>,"{""aws"": {""bucket_name"": ""nasa-irsa-euclid-q1"",...",,1


In [11]:
filename = df_im_euclid['access_url'].to_list()[2]
filesize = df_im_euclid['access_estsize'].to_list()[2]/1000000
print(filename,filesize)
fname = download_file(filename, cache=True)
hdu_mer_irsa = fits.open(fname)
print(hdu_mer_irsa.info())

head_mer_irsa = hdu_mer_irsa[0].header

https://irsa.ipac.caltech.edu/ibe/data/euclid/q1/MER/102044185/NISP/EUC_MER_BGSUB-MOSAIC-NIR-Y_TILE102044185-15D02F_20241021T005729.282854Z_00.00.fits 1.474566
Filename: /Users/shemmati/.astropy/cache/download/url/a67f0c08a13a1e8b07cbf8f268635edd/contents
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      48   (19200, 19200)   float32   
None


In [12]:

download_path = '../data/JWST_JADES/'
hdu_mer_irsa.writeto(os.path.join(download_path, 'MER_image_VIS_2.fits'), overwrite=True)

# JWST NIRCAM:

In [13]:
#https://archive.stsci.edu/hlsp/jades

from astroquery.mast import Observations
f115w_obs = Observations.query_criteria(provenance_name="jades",instrument_name='NIRCAM/IMAGE',filters='F115W')
data_products = Observations.get_product_list(f115w_obs)
Observations.download_products(data_products)

Local Path,Status,Message,URL
str122,str8,object,object
./mastDownload/HLSP/hlsp_jades_jwst_nircam_goods-s-deep_f115w_v2.0/hlsp_jades_jwst_nircam_goods-s-deep_f115w_v2.0_drz.fits,COMPLETE,None,None
./mastDownload/HLSP/hlsp_jades_jwst_nircam_goods-n_f115w_v1.0/hlsp_jades_jwst_nircam_goods-n_f115w_v1.0_drz.fits,COMPLETE,None,None
